# **🚚 Automated ETL Pipeline: Preprocessing Global Logistics Data with Python**

---



---



# Dataset Content

This dataset captures logistics and supply chain operations collected from a logistics network in Southern California, covering transportation, warehouse activity, routing conditions, and real-time monitoring signals (GPS/IoT/WMS/external sources). The file is organized as hourly records.
Rows: 32,065 • Columns: 26

1. Tracking & Temporal Data
timestamp: The specific date and hour of the recorded event.

vehicle_gps_latitude/longitude: Precise geographic coordinates for route mapping.

eta_variation_hours: The deviation between estimated and actual arrival times.

lead_time_days: Total duration from order placement to supplier delivery.

2. Operational Efficiency & Financials
fuel_consumption_rate: Liters consumed per hour (key for sustainability analysis).

shipping_costs: Total transportation expenditure in USD.

loading_unloading_time: Time spent at hubs, indicating bottleneck locations.

warehouse_inventory_level: Current stock units available at the storage site.

3. IoT & Condition Monitoring
iot_temperature: Real-time thermal tracking (critical for cold-chain goods).

cargo_condition_status: A sensor-based score (0-1) reflecting the physical integrity of the goods.

4. Environmental & External Risks
traffic_congestion_level: Road density score (0-10).

port_congestion_level: Operational delays at maritime or air gateways (0-10).

weather_condition_severity: Impact of climate events on transit safety (0-1).

route_risk_level: A combined safety score for the specific path taken (0-10).

5. Behavioral & Reliability Metrics
driver_behavior_score: Measures safe driving practices (0-1).

fatigue_monitoring_score: Predicts driver exhaustion levels based on biometric or telematics data.

supplier_reliability_score: Historical performance rating of the vendor.

6. Predictive Labels (KPIs)
disruption_likelihood_score: Probability of a major supply chain break.

delay_probability: Estimated chance of a late delivery.

risk_classification: Categorical label (Low, Moderate, High Risk).


## **1.Data Inspection**

In [4]:
import pandas as pd
import numpy as np
df = pd.read_csv('/content/sample_data/dynamic_supply_chain_logistics_dataset.csv')

In [5]:
df.head()

,timestamp,vehicle_gps_latitude,vehicle_gps_longitude,fuel_consumption_rate,eta_variation_hours,traffic_congestion_level,warehouse_inventory_level,loading_unloading_time,handling_equipment_availability,order_fulfillment_status,...,iot_temperature,cargo_condition_status,route_risk_level,customs_clearance_time,driver_behavior_score,fatigue_monitoring_score,disruption_likelihood_score,delay_probability,risk_classification,delivery_time_deviation
0,2021-01-01 00:00:00,40.375568,-77.014318,5.136512,4.998009,5.927586,985.716862,4.951392,0.481294,0.761166,...,0.574400,0.777263,1.182116,0.502006,0.033843,0.978599,0.506152,0.885291,Moderate Risk,9.110682
1,2021-01-01 01:00:00,33.507818,-117.036902,5.101512,0.984929,1.591992,396.700206,1.030379,0.620780,0.196594,...,-9.753493,0.091839,9.611988,0.966774,0.201725,0.918586,0.980784,0.544178,High Risk,8.175281
2,2021-01-01 02:00:00,30.020640,-75.269224,5.090803,4.972665,8.787765,832.408935,4.220229,0.810933,0.152742,...,-6.491034,0.253529,6.570431,0.945627,0.264045,0.394215,0.998633,0.803322,High Risk,1.283594
3,2021-01-01 03:00:00,36.649223,-70.190529,8.219558,3.095064,0.045257,0.573283,0.530186,0.008525,0.811885,...,-0.151276,0.877576,0.548952,4.674035,0.362885,0.905444,0.993320,0.025977,High Risk,9.304897
4,2021-01-01 04:00:00,30.001279,-70.012195,5.000075,3.216077,8.004851,914.925067,3.620890,0.020083,0.053659,...,2.429448,0.262081,8.861443,3.445429,0.016957,0.258702,0.912433,0.991122,High Risk,7.752484


In [6]:
df.shape

(32065, 26)

### DataFrame Info



In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32065 entries, 0 to 32064
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   timestamp                        32065 non-null  object 
 1   vehicle_gps_latitude             32065 non-null  float64
 2   vehicle_gps_longitude            32065 non-null  float64
 3   fuel_consumption_rate            32065 non-null  float64
 4   eta_variation_hours              32065 non-null  float64
 5   traffic_congestion_level         32065 non-null  float64
 6   warehouse_inventory_level        32065 non-null  float64
 7   loading_unloading_time           32065 non-null  float64
 8   handling_equipment_availability  32065 non-null  float64
 9   order_fulfillment_status         32065 non-null  float64
 10  weather_condition_severity       32065 non-null  float64
 11  port_congestion_level            32065 non-null  float64
 12  shipping_costs    

### DataFrame Summary Statistics



In [ ]:
df.describe()

,vehicle_gps_latitude,vehicle_gps_longitude,fuel_consumption_rate,eta_variation_hours,traffic_congestion_level,warehouse_inventory_level,loading_unloading_time,handling_equipment_availability,order_fulfillment_status,weather_condition_severity,...,historical_demand,iot_temperature,cargo_condition_status,route_risk_level,customs_clearance_time,driver_behavior_score,fatigue_monitoring_score,disruption_likelihood_score,delay_probability,delivery_time_deviation
count,32065.000000,32065.000000,32065.000000,32065.000000,3.206500e+04,3.206500e+04,32065.000000,3.206500e+04,32065.000000,3.206500e+04,...,32065.000000,32065.000000,3.206500e+04,32065.000000,32065.000000,3.206500e+04,3.206500e+04,32065.000000,32065.000000,32065.000000
mean,38.023589,-90.116648,8.011735,2.893068,4.991493e+00,2.992547e+02,2.291669,3.026954e-01,0.600740,4.976082e-01,...,6022.001286,0.044792,2.972816e-01,7.001144,2.296448,4.983913e-01,6.008723e-01,0.803656,0.699077,5.177648
std,6.917909,17.369244,4.264960,2.274044,3.532048e+00,3.234435e+02,1.554202,3.259146e-01,0.345672,3.532853e-01,...,3427.638017,14.187486,3.216115e-01,3.236328,1.555932,3.541589e-01,3.458101e-01,0.279185,0.324514,4.157988
min,30.000000,-119.999998,5.000000,-1.999993,1.091633e-09,1.322210e-12,0.500000,4.565769e-16,0.000001,4.536949e-09,...,100.002966,-10.000000,7.255415e-19,0.000050,0.500000,4.043927e-09,3.269508e-07,0.000048,0.000003,-1.999998
25%,31.280550,-106.253913,5.019984,1.185744,1.474720e+00,1.605163e+01,0.774798,1.710828e-02,0.277096,1.440135e-01,...,2822.607616,-9.931074,1.678269e-02,4.593407,0.776166,1.443567e-01,2.783148e-01,0.693739,0.456009,1.269197
50%,36.413820,-86.293414,5.636036,3.882059,4.981244e+00,1.572880e+02,1.917121,1.595151e-01,0.680553,4.961781e-01,...,6785.123209,-7.858681,1.549760e-01,8.385605,1.938273,4.988468e-01,6.831130e-01,0.958128,0.839599,6.113662
75%,44.453655,-73.079367,9.669944,4.884355,8.534902e+00,5.405980e+02,3.734188,5.535954e-01,0.938160,8.498226e-01,...,9374.252913,6.024012,5.405408e-01,9.836152,3.750817,8.510762e-01,9.372889e-01,0.998746,0.982391,9.249206
max,50.000000,-70.000000,19.999875,5.000000,9.999999e+00,9.999993e+02,5.000000,9.999995e-01,1.000000,1.000000e+00,...,10000.000000,39.999886,1.000000e+00,10.000000,5.000000,1.000000e+00,1.000000e+00,1.000000,1.000000,10.000000


<div style="text-align:center; background-color:#D3D3D3; padding:20px;">
<h1 style="font-size:36px; color:#fffff;"><b> 2.Convert Data Types </b></h1>
</div>


In [7]:
#  Convert 'timestamp' to datetime objects
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [8]:
# Clean and Standardize Text
df['risk_classification'] = df['risk_classification'].str.lower()
df['risk_classification'] = df['risk_classification'].str.strip().replace(r'\s+', ' ', regex=True)
df['risk_classification'] = df['risk_classification'].str.replace(r'[^\w\s]', '', regex=True)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32065 entries, 0 to 32064
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        32065 non-null  datetime64[ns]
 1   vehicle_gps_latitude             32065 non-null  float64       
 2   vehicle_gps_longitude            32065 non-null  float64       
 3   fuel_consumption_rate            32065 non-null  float64       
 4   eta_variation_hours              32065 non-null  float64       
 5   traffic_congestion_level         32065 non-null  float64       
 6   warehouse_inventory_level        32065 non-null  float64       
 7   loading_unloading_time           32065 non-null  float64       
 8   handling_equipment_availability  32065 non-null  float64       
 9   order_fulfillment_status         32065 non-null  float64       
 10  weather_condition_severity       32065 non-null  float64  

<div style="text-align:center; background-color:#D3D3D3; padding:20px;">
  <h1 style="font-size:36px; color:#fffff;"><b> 3.Filter last 6 months </b></h1>
</div>


In [10]:
analysis_end_date = df['timestamp'].max()
analysis_start_date = analysis_end_date - pd.DateOffset(months=6)
df = df[df['timestamp'] >= analysis_start_date].copy()

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4369 entries, 27696 to 32064
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        4369 non-null   datetime64[ns]
 1   vehicle_gps_latitude             4369 non-null   float64       
 2   vehicle_gps_longitude            4369 non-null   float64       
 3   fuel_consumption_rate            4369 non-null   float64       
 4   eta_variation_hours              4369 non-null   float64       
 5   traffic_congestion_level         4369 non-null   float64       
 6   warehouse_inventory_level        4369 non-null   float64       
 7   loading_unloading_time           4369 non-null   float64       
 8   handling_equipment_availability  4369 non-null   float64       
 9   order_fulfillment_status         4369 non-null   float64       
 10  weather_condition_severity       4369 non-null   float64    

<div style="text-align:center; background-color:#D3D3D3; padding:20px;">
  <h1 style="font-size:36px; color:#fffff;"><b> 4.Add Custom Columns </b></h1>
</div>


In [20]:
# Extract year ,month,day and hour from timestamp
df['Date'] = df['timestamp'].dt.date
df['Year'] = df['timestamp'].dt.year
df['Month'] = df['timestamp'].dt.month
df['Month_Name'] = df['timestamp'].dt.month_name()
df['Week'] = df['timestamp'].dt.isocalendar().week
df['Day_Name'] = df['timestamp'].dt.day_name()
df['Hour'] = df['timestamp'].dt.hour
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4369 entries, 27696 to 32064
Data columns (total 38 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        4369 non-null   datetime64[ns]
 1   vehicle_gps_latitude             4369 non-null   float64       
 2   vehicle_gps_longitude            4369 non-null   float64       
 3   fuel_consumption_rate            4369 non-null   float64       
 4   eta_variation_hours              4369 non-null   float64       
 5   traffic_congestion_level         4369 non-null   float64       
 6   warehouse_inventory_level        4369 non-null   float64       
 7   loading_unloading_time           4369 non-null   float64       
 8   handling_equipment_availability  4369 non-null   float64       
 9   order_fulfillment_status         4369 non-null   float64       
 10  weather_condition_severity       4369 non-null   float64    

In [21]:
# Create Geo Zone Columns (Rounding Logic)
df['Lat_Round'] = df['vehicle_gps_latitude'].round(1)
df['Lon_Round'] = df['vehicle_gps_longitude'].round(1)
df['Geo_Zone'] = df['Lat_Round'].astype(str) + "," + df['Lon_Round'].astype(str)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4369 entries, 27696 to 32064
Data columns (total 38 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        4369 non-null   datetime64[ns]
 1   vehicle_gps_latitude             4369 non-null   float64       
 2   vehicle_gps_longitude            4369 non-null   float64       
 3   fuel_consumption_rate            4369 non-null   float64       
 4   eta_variation_hours              4369 non-null   float64       
 5   traffic_congestion_level         4369 non-null   float64       
 6   warehouse_inventory_level        4369 non-null   float64       
 7   loading_unloading_time           4369 non-null   float64       
 8   handling_equipment_availability  4369 non-null   float64       
 9   order_fulfillment_status         4369 non-null   float64       
 10  weather_condition_severity       4369 non-null   float64    

In [22]:
# Operational Bands
def get_band(val, low, high):
    if val <= low: return "Low"
    elif val <= high: return "Medium"
    else: return "High"

df['Traffic_Band'] = df['traffic_congestion_level'].apply(lambda x: get_band(x, 3, 7))
df['RouteRisk_Band'] = df['route_risk_level'].apply(lambda x: get_band(x, 3, 7))
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4369 entries, 27696 to 32064
Data columns (total 38 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        4369 non-null   datetime64[ns]
 1   vehicle_gps_latitude             4369 non-null   float64       
 2   vehicle_gps_longitude            4369 non-null   float64       
 3   fuel_consumption_rate            4369 non-null   float64       
 4   eta_variation_hours              4369 non-null   float64       
 5   traffic_congestion_level         4369 non-null   float64       
 6   warehouse_inventory_level        4369 non-null   float64       
 7   loading_unloading_time           4369 non-null   float64       
 8   handling_equipment_availability  4369 non-null   float64       
 9   order_fulfillment_status         4369 non-null   float64       
 10  weather_condition_severity       4369 non-null   float64    

In [23]:
#  Binary & Delay Flags
df['Equipment_Available_Flag'] = (df['handling_equipment_availability'] >= 0.5).astype(int)
df['OnTime_Fulfillment_Flag'] = (df['order_fulfillment_status'] >= 0.5).astype(int)
df['Cargo_Good_Flag'] = (df['cargo_condition_status'] >= 0.5).astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4369 entries, 27696 to 32064
Data columns (total 41 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        4369 non-null   datetime64[ns]
 1   vehicle_gps_latitude             4369 non-null   float64       
 2   vehicle_gps_longitude            4369 non-null   float64       
 3   fuel_consumption_rate            4369 non-null   float64       
 4   eta_variation_hours              4369 non-null   float64       
 5   traffic_congestion_level         4369 non-null   float64       
 6   warehouse_inventory_level        4369 non-null   float64       
 7   loading_unloading_time           4369 non-null   float64       
 8   handling_equipment_availability  4369 non-null   float64       
 9   order_fulfillment_status         4369 non-null   float64       
 10  weather_condition_severity       4369 non-null   float64    

In [25]:
# Weather Banding (Specific Thresholds)
df['Weather_Band'] = pd.cut(df['weather_condition_severity'],
                            bins=[0, 0.33, 0.66, 1.0],
                            labels=['Mild', 'Moderate', 'Severe'],
                            include_lowest=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4369 entries, 27696 to 32064
Data columns (total 42 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        4369 non-null   datetime64[ns]
 1   vehicle_gps_latitude             4369 non-null   float64       
 2   vehicle_gps_longitude            4369 non-null   float64       
 3   fuel_consumption_rate            4369 non-null   float64       
 4   eta_variation_hours              4369 non-null   float64       
 5   traffic_congestion_level         4369 non-null   float64       
 6   warehouse_inventory_level        4369 non-null   float64       
 7   loading_unloading_time           4369 non-null   float64       
 8   handling_equipment_availability  4369 non-null   float64       
 9   order_fulfillment_status         4369 non-null   float64       
 10  weather_condition_severity       4369 non-null   float64    

In [26]:
# Delay Flags
df['Late_Flag'] = (df['delivery_time_deviation'] > 0).astype(int)
df['HighDelayRisk_Flag'] = (df['delay_probability'] >= 0.5).astype(int)
df['Disruption_Flag'] = (df['disruption_likelihood_score'] >= 0.5).astype(int)
df['Disruption_Flag'] = (df['disruption_likelihood_score']>=0.5).astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4369 entries, 27696 to 32064
Data columns (total 45 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        4369 non-null   datetime64[ns]
 1   vehicle_gps_latitude             4369 non-null   float64       
 2   vehicle_gps_longitude            4369 non-null   float64       
 3   fuel_consumption_rate            4369 non-null   float64       
 4   eta_variation_hours              4369 non-null   float64       
 5   traffic_congestion_level         4369 non-null   float64       
 6   warehouse_inventory_level        4369 non-null   float64       
 7   loading_unloading_time           4369 non-null   float64       
 8   handling_equipment_availability  4369 non-null   float64       
 9   order_fulfillment_status         4369 non-null   float64       
 10  weather_condition_severity       4369 non-null   float64    

In [27]:
# Agg_Daily
agg_daily = df.groupby(['Date', 'Geo_Zone']).agg({
    'shipping_costs': 'mean',
    'delay_probability': 'mean',
    'OnTime_Fulfillment_Flag': 'mean',
    'eta_variation_hours': 'mean',
    'traffic_congestion_level': 'mean'
}).reset_index()
agg_daily.columns = ['Date', 'Geo_Zone', 'Avg_Shipping_Costs', 'Avg_Delay_Prob', 'OnTime_Rate', 'Avg_ETA_Var', 'Avg_Traffic']

<class 'pandas.core.frame.DataFrame'>
Index: 4369 entries, 27696 to 32064
Data columns (total 45 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        4369 non-null   datetime64[ns]
 1   vehicle_gps_latitude             4369 non-null   float64       
 2   vehicle_gps_longitude            4369 non-null   float64       
 3   fuel_consumption_rate            4369 non-null   float64       
 4   eta_variation_hours              4369 non-null   float64       
 5   traffic_congestion_level         4369 non-null   float64       
 6   warehouse_inventory_level        4369 non-null   float64       
 7   loading_unloading_time           4369 non-null   float64       
 8   handling_equipment_availability  4369 non-null   float64       
 9   order_fulfillment_status         4369 non-null   float64       
 10  weather_condition_severity       4369 non-null   float64    

In [31]:
# Creating Dim_Date
dim_date = df[['Date', 'Year', 'Month', 'Month_Name']].drop_duplicates()

## <div style="text-align:center; background-color:#D3D3D3; padding:20px;">
  <h1 style="font-size:36px; color:#fffff;"><b> 5.Extract Cleaned Files </b></h1>
</div>


In [32]:
df.to_csv('Fact_Operations_Cleaned.csv', index=False)
agg_daily.to_csv('Agg_Daily_Summary.csv', index=False)
dim_date.to_csv('Dim_Date.csv', index=False)
